In [ ]:
%matplotlib inline
import sys
sys.path.append("../..")

import numpy as np
import scipy

from pypanodecoder import optical_model

import scipy.optimize

In [ ]:
# Define properties of ideal lens: focal length, diameter and refractive
# index at referenece wavelength at which polynomial is optimized 
F = 60.78
D = 46.09
wl_ref = 546
n_ref = 1.4820583082771652
r = F*(n_ref-1)

In [ ]:
# Setup initial polynomial. Use spherical approximation
ni = np.arange(10.0)
p0 = np.append([0],(-1.0)**ni*scipy.special.gamma(2*ni+1)/(scipy.special.gamma(ni+1)*scipy.special.gamma(ni+2)))
p0 *= 2.0/2.0**(2.0*np.arange(len(p0)))

In [ ]:
# We only optimize parameters 2 to N
# p[0] is not relevant as polynomial defines normal only (through its derivative)
# p[1] defines the focal length for rays near the axis and is fixed by F
# The px function generates the full polynomial from the parameters being optimized
x0 = p0[2:]
def px(x, px1=p0[1], rx=r):
    px = np.append(np.asarray([0, px1]), x)
    px *= rx/rx**(2.0*np.arange(len(px)))
    return px

In [ ]:
# Raytracer defined through optical model dictionary. Here we set one
# up based on polynomial being evaluated
def make_optical_model(p):
    return  {
        "F":             F,
        "D":             D,
        "thickness":     0.18,             # Not used
        "groove_width":  0.0508,           # Not used
        "draft_angle":   3.0,              # Not used
        "ev_ref":        1239.842/wl_ref,
        "nm_ref":        wl_ref,           # Not used
        "n_ref":         n_ref,            # Not used
        "p_out":         p,
        "roughness":     0,
        "npixel":        32,               # Not used
        "pixel_spacing": 0.32,             # Not used
        "pixel_active":  0.30              # Not used
    }

In [ ]:
# Calculate PSF by tracing a bundle of rays defined on a square grid through
# the optical model and calculate spot squared
def calc_psf(p, theta = 0, n = n_ref, num_rays=1000000):
    optics = make_optical_model(p)
    datapack = {
        "optical_model": optics
    }

    direction = np.asarray([np.sin(np.deg2rad(theta)),-np.cos(np.deg2rad(theta)),0.0])

    bundle = optical_model.trace_parallel_ray_bundle(direction, num_rays, datapack, 
        thick_lens=False, zn=0, focal_offset=0.0, energy_eV=optics['ev_ref'], use_grid=True, fixed_n=n)

    mask = bundle.valid

    xfp = bundle.pos[mask,0]
    zfp = bundle.pos[mask,2]

    xmean = np.mean(xfp)
    zmean = np.mean(zfp)

    return np.mean((xfp-xmean)**2 + (zfp-zmean)**2)

In [ ]:
# Print initial spot size (in cm)
print(f'Initial PSF: {np.sqrt(calc_psf(px(x0))):.10f}')

In [ ]:
# Run the optimizer
def psf(x):
    psfval = calc_psf(px(x))
    print(f'Iteration PSF: {np.sqrt(psfval):.10f}',' '*10,end='\r')
    return psfval
optres = scipy.optimize.minimize(psf, x0)

In [ ]:
# Print the fitting status and results
print(optres)

In [ ]:
# Print the fitted polynomial parameters
print("Final polynomial:",px(optres.x))

In [ ]:
# And the final model (which could be saved as a JSON)
print(make_optical_model(px(optres.x)))